# Realistic Operational Data

The `01_operational_data` notebook generates a synthetic supply chain network with independently random parameters (profit margin, inventory, demand, capacity all drawn uniformly at random), and every stress test in `02`/`03` disrupts a single random node. That's a good way to learn the time-to-recover (TTR) / time-to-survive (TTS) methodology, but it doesn't tell you much about how a *real* supply chain behaves under stress.

This notebook generates three additional tiers of realism, all using the exact same optimization engine from `scripts/utils.py` (unmodified):

| Tier | Scale | What's different |
|---|---|---|
| **Simple** | ~27 nodes, single product line | Fictional (illustrative) company names, BOM-propagated inventory/capacity instead of pure randomness |
| **Medium** | ~700 nodes, multi-product-line | Adds region-tagged suppliers (concentrated in real-world sourcing geographies) so regional disruptions are meaningful |
| **Complex** | ~2,300 nodes each | Two networks anchored on **real, publicly-known companies** — an Nvidia-like AI/GPU chip supply chain and an Apple-like consumer electronics supply chain |

See the README's "Realistic Stress-Test Scenario Ladder" section for the full methodology and citations. Notebooks `06` and `07` run the stress tests on these datasets.

## Cluster Configuration
This notebook was tested on the following Databricks cluster configuration:
- **Databricks Runtime Version:** 17.3 LTS ML (includes Apache Spark 4.0.0, Scala 2.13)
- **Single Node**
    - Azure: Standard_DS4_v2 (28 GB Memory, 8 Cores)
    - AWS: m5d.2xlarge (32 GB Memory, 8 Cores)
- **Photon Acceleration:** Disabled (Photon boosts Apache Spark workloads; not all ML workloads will see an improvement)

In [0]:
%pip install -r ./requirements.txt --quiet
dbutils.library.restartPython()

In [ ]:
import json
import os
import scripts.utils as utils
import scripts.realistic_topologies as rt

We will store the generated datasets in the same Unity Catalog Volume used by `01_operational_data`.

In [0]:
catalog = "supply_chain_stress_test"  # Change here
schema = "data"                       # Change here
volume = "operational"                # Change here

# Make sure that the catalog, the schema and the volume exist
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
_ = spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

In [ ]:
dbutils.widgets.dropdown("regenerate_data", "no", ["no", "yes"], "Regenerate data")
regenerate_data = dbutils.widgets.get("regenerate_data") == "yes"

volume_dir = f"/Volumes/{catalog}/{schema}/{volume}"
dataset_filenames = {
    "simple": "dataset_realistic_simple.json",
    "medium": "dataset_realistic_medium.json",
    "nvidia": "dataset_realistic_nvidia.json",
    "apple": "dataset_realistic_apple.json",
    "mps": "dataset_realistic_mps.json",
    "nvidia_planet": "dataset_realistic_nvidia_planet.json",
    "apple_planet": "dataset_realistic_apple_planet.json",
}
datasets_exist = all(
    os.path.exists(f"{volume_dir}/{filename}") for filename in dataset_filenames.values()
)
skip_generation = datasets_exist and not regenerate_data
print(f"skip_generation={skip_generation} (regenerate_data={regenerate_data}, datasets_exist={datasets_exist})")

Every generator below is deterministically seeded (see `scripts/realistic_topologies.py`), so re-running this notebook produces byte-identical datasets — regenerating is redundant work, not a correctness concern. The `regenerate_data` widget (default `"no"`) skips generation and loads the existing files from the volume when they're already there; set it to `"yes"` (via the notebook UI or a job's `base_parameters`) to force a fresh regenerate-and-overwrite.

## Simple Network

A small, single-product-line illustrative manufacturer (fictional company names — see `scripts/company_profiles.py`). Unlike `generate_data`, inventory and capacity are sized from BOM-propagated demand and a per-node criticality tag (derived from how many alternate suppliers exist for each material), so a handful of sole-sourced nodes end up genuinely risk-bearing instead of every parameter being an independent coin flip.

In [ ]:
if skip_generation:
    with open(f"{volume_dir}/{dataset_filenames['simple']}") as f:
        simple = json.load(f)
else:
    simple = rt.generate_simple_network()
print(f"tier1={len(simple['tier1'])}, tier2={len(simple['tier2'])}, tier3={len(simple['tier3'])}")

In [0]:
utils.visualize_network(simple)

## Medium Network

A ~700-node, multi-product-line manufacturer. Suppliers are tagged with a `region` (concentrated in a few real-world auto-parts sourcing geographies — China, Mexico, USA, Germany, Vietnam, Thailand), which is what makes the regional/correlated disruption scenarios in notebook `06` meaningful. `generate_data`'s output has no such tag.

In [ ]:
if skip_generation:
    with open(f"{volume_dir}/{dataset_filenames['medium']}") as f:
        medium = json.load(f)
else:
    medium = rt.generate_medium_network()
print(f"tier1={len(medium['tier1'])}, tier2={len(medium['tier2'])}, tier3={len(medium['tier3'])}")

At ~700 nodes the tier-scatter plot from `visualize_network` gets too dense to read. A region breakdown is more informative at this scale.

In [0]:
import pandas as pd

pd.Series(medium["region"]).value_counts().plot(
    kind="bar", title="Medium network: supplier count by region", figsize=(8, 4)
)

## Complex Networks: Nvidia-like and Apple-like

These two ~2,300-node networks anchor tier 2 (direct component/module suppliers) and tier 3 (their equipment/raw-material suppliers) on **real, publicly-known companies** — e.g. TSMC, SK Hynix, ASML, Foxconn — drawn from public secondary reporting about these companies' real supplier relationships (see `scripts/company_profiles.py` for the full list and citations). Each real anchor is fanned out with synthetic "rest of market" peers and synthetic sub-suppliers to reach a realistic scale.

> **Disclaimer:** All company names referenced below are drawn from public secondary reporting, current as of 2026-07. **Every numeric figure attached to a node in this accelerator — profit margin, inventory, demand, capacity — is synthetic and illustrative.** None of these figures represent actual disclosed financial or operational data from Nvidia, Apple, or any named supplier, and must not be used to draw real inferences about those companies.

In [ ]:
if skip_generation:
    with open(f"{volume_dir}/{dataset_filenames['nvidia']}") as f:
        nvidia = json.load(f)
else:
    nvidia = rt.generate_complex_network("nvidia")
print(f"tier1={len(nvidia['tier1'])}, tier2={len(nvidia['tier2'])}, tier3={len(nvidia['tier3'])}")

`visualize_network`'s scatter plot doesn't scale to thousands of nodes. `visualize_anchor_backbone` instead draws only the named real companies and labels each with how many synthetic suppliers feed into it.

In [0]:
rt.visualize_anchor_backbone(nvidia)

In [ ]:
if skip_generation:
    with open(f"{volume_dir}/{dataset_filenames['apple']}") as f:
        apple = json.load(f)
else:
    apple = rt.generate_complex_network("apple")
print(f"tier1={len(apple['tier1'])}, tier2={len(apple['tier2'])}, tier3={len(apple['tier3'])}")

In [0]:
rt.visualize_anchor_backbone(apple)

## Complex Network: MPS-like (fabless power-management maker)

A third ~2,300-node complex network anchored on **Monolithic Power Systems (MPS)**, a fabless PMIC / power-module maker. Its back-end structure is distinctive: MPS contracts foundries (anchored on **Vanguard International Semiconductor**) for its proprietary BCD-process wafers, then ships them to its **wholly-owned Chengdu wafer-sort/final-test facility**, with packaging outsourced to **OSAT partners in China and Malaysia** and a **Penang engineering hub**. See `scripts/company_profiles.py` for the full anchor list, per-anchor citations, and the modeling simplifications (the captive Chengdu/Penang facilities are modeled as tier-2 nodes; the OSAT partners use generic labels since the source doc doesn't name them; ASML is deliberately excluded since MPS's mature-node BCD process doesn't use EUV lithography).

This MPS dataset is what notebook `09_mps_cuopt_pipeline` layers its FJSP (back-end scheduling), CVRPTW (distribution routing), and MEIO (multi-echelon inventory) optimization problems on top of.

> **Disclaimer:** All company names are drawn from public secondary reporting + the `MPS Supply Chain Optimization Pipeline.md` design doc, current as of 2026-07. **Every numeric figure attached to a node — profit margin, inventory, demand, capacity — is synthetic and illustrative.** None represent actual disclosed financial or operational data from MPS, VIS, or any named supplier.

In [ ]:
if skip_generation:
    with open(f"{volume_dir}/{dataset_filenames['mps']}") as f:
        mps = json.load(f)
else:
    mps = rt.generate_complex_network("mps")
print(f"tier1={len(mps['tier1'])}, tier2={len(mps['tier2'])}, tier3={len(mps['tier3'])}")
rt.visualize_anchor_backbone(mps)

## Optional: Planet-Scale Demo

`generate_complex_network` also accepts a `scale_factor` that proportionally scales every real anchor's synthetic peer/child fan-out, so the network grows without diluting toward generic padding. `generate_complex_network_at_scale(company, scale="planet")` uses a tuned preset (~76,000 total nodes) from `SCALE_PRESETS` in `scripts/realistic_topologies.py`. See the README's "Scaling to planet-scale networks" section for the measured generation/solve-time tradeoffs — generation stays fast at this scale, but LP solve time per scenario grows with total node count since the (unmodified) LP re-solves the whole network every time.

This cell is optional and not required for `06`/`07` to run — it exists to demonstrate the scale-up path. `06`/`07` continue to use the `nvidia`/`apple` datasets written below.

In [ ]:
if skip_generation:
    with open(f"{volume_dir}/{dataset_filenames['nvidia_planet']}") as f:
        nvidia_planet = json.load(f)
    with open(f"{volume_dir}/{dataset_filenames['apple_planet']}") as f:
        apple_planet = json.load(f)
else:
    nvidia_planet = rt.generate_complex_network_at_scale("nvidia", scale="planet")
    apple_planet = rt.generate_complex_network_at_scale("apple", scale="planet")
print(f"nvidia planet-scale: tier1={len(nvidia_planet['tier1'])}, tier2={len(nvidia_planet['tier2'])}, tier3={len(nvidia_planet['tier3'])}")
print(f"apple planet-scale: tier1={len(apple_planet['tier1'])}, tier2={len(apple_planet['tier2'])}, tier3={len(apple_planet['tier3'])}")

## Write Datasets

Let's write all seven datasets as JSON files, alongside the existing `dataset_small.json`/`dataset_large.json` (which are left untouched). Skipped entirely when `skip_generation` is `True` — the files loaded above are already what's on disk.

In [ ]:
if skip_generation:
    print("skip_generation=True — datasets loaded from the volume, nothing to write.")
else:
    datasets = {
        "dataset_realistic_simple.json": simple,
        "dataset_realistic_medium.json": medium,
        "dataset_realistic_nvidia.json": nvidia,
        "dataset_realistic_apple.json": apple,
        "dataset_realistic_mps.json": mps,
        "dataset_realistic_nvidia_planet.json": nvidia_planet,
        "dataset_realistic_apple_planet.json": apple_planet,
    }
    for filename, dataset in datasets.items():
        with open(f"{volume_dir}/{filename}", "w") as json_file:
            json.dump(dataset, json_file)
    print(f"Wrote {len(datasets)} dataset files to {volume_dir}")

## Wrap Up

In this notebook, we generated seven realistic supply-chain networks spanning simple, medium, complex (Nvidia-like / Apple-like / MPS-like), and planet-scale (Nvidia-like / Apple-like, ~76,000 nodes each) — all consumable by the exact same `build_and_solve_ttr`/`build_and_solve_tts` optimization engine used throughout this accelerator. In the next notebooks, `06_realistic_stress_testing (simple and medium)` and `07_realistic_stress_testing (complex network)`, we run a named library of disruption scenarios (single-supplier, regional, and material-wide shortage — including several inspired by real historical events) against these datasets. Notebook `09_mps_cuopt_pipeline` goes further with the MPS-like network, deriving GPU-accelerated scheduling (FJSP) and routing (CVRPTW) problems plus a CPU-solved multi-echelon inventory (MEIO) model on top of it.

&copy; 2025 Databricks, Inc. All rights reserved. The source in this notebook is provided subject to the Databricks License [https://databricks.com/db-license-source].  All included or referenced third party libraries are subject to the licenses set forth below.

| library                                | description             | license    | source                                              |
|----------------------------------------|-------------------------|------------|-----------------------------------------------------|
| pyomo | An object-oriented algebraic modeling language in Python for structured optimization problems | BSD-3 | https://pypi.org/project/pyomo/
| highspy | Linear optimization solver (HiGHS) | MIT | https://pypi.org/project/highspy/
| ray | Framework for scaling AI/Python applications | Apache 2.0 | https://github.com/ray-project/ray